# Atlas Power Comparison

Compare v2.3.0 and v3.1.0 power analyses for harmonized notochordal sheath cells at 36 hpf.

This notebook is focused on the pre-specified sheath-cell endpoint. Broad all-cell-type screen behavior, including larger BH correction universes, should be evaluated in a separate notebook.

This notebook uses the same plain R Dirichlet simulation and NB GLM inference machinery as `power_test_notebook.ipynb`. The target cell type is parameterized so other harmonized populations can be swapped in by changing `target_standardized_cell_types`.

The simulation model is fit to harmonized cell counts normalized against total recovered cells. Internally, the modeled notochord/hypochord groups are paired with an explicit `other cells` category so each embryo sums to its full recovered-cell count. Simulated embryo depths are sampled with replacement from the observed 36 hpf total recovered-cell distribution for each atlas, and cell counts are sampled multinomially from Dirichlet-simulated proportions.

The expensive simulation sweep is guarded by `run_power_sweep`. With the default `FALSE`, setup and observed-count plots can run without launching simulations.

In [ ]:
library(MASS)
library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)
library(purrr)
library(stringr)
library(VGAM)

theme_set(theme_bw(base_size = 18))

In [ ]:
# ------------------------------------------------------------------
# User-editable parameters
# ------------------------------------------------------------------
out_dir <- "/net/trapnell/vol1/home/nlammers/projects/data/morphseq/results/nlammers/20260615"
count_cache_dir <- file.path(out_dir, "atlas_count_cache")
power_out_dir <- file.path(out_dir, "atlas_power_comparison")
dir.create(power_out_dir, recursive = TRUE, showWarnings = FALSE)

datasets_to_compare <- c("v2.3.0", "v3.1.0")
analysis_timepoint <- 24

# Leave NULL to include all harmonized notochord/hypochord groups in the model.
# Set to a character vector to restrict the inference universe.
analysis_standardized_cell_groups <- NULL

target_standardized_cell_types <- c("notochordal sheath cell")

# Full recovered cells per simulated embryo. These are not notochord-only.
# Pull the dataset-specific median total recovered cells from the atlas comparison output.
comparison_cell_counts_path <- file.path(out_dir, "embryo_cell_counts_long.csv")

derive_cells_per_embryo_by_dataset <- function(
  comparison_path = comparison_cell_counts_path,
  dataset_names = datasets_to_compare,
  timepoint = analysis_timepoint
) {
  if (!file.exists(comparison_path)) {
    stop("Missing atlas comparison count table: ", comparison_path)
  }

  comparison_counts <- read.csv(comparison_path, stringsAsFactors = FALSE)
  required_cols <- c("dataset", "embryo", "hpf", "cell_count")
  missing_cols <- setdiff(required_cols, colnames(comparison_counts))
  if (length(missing_cols) > 0) {
    stop("Missing required column(s) in ", comparison_path, ": ", paste(missing_cols, collapse = ", "))
  }

  depth_table <- comparison_counts %>%
    filter(dataset %in% dataset_names, hpf == timepoint) %>%
    group_by(dataset, embryo) %>%
    summarise(total_recovered_cells = sum(cell_count, na.rm = TRUE), .groups = "drop") %>%
    group_by(dataset) %>%
    summarise(
      cells_per_embryo = round(median(total_recovered_cells, na.rm = TRUE)),
      median_total_recovered_cells = median(total_recovered_cells, na.rm = TRUE),
      n_embryos_for_depth = n(),
      source_hpf = timepoint,
      .groups = "drop"
    ) %>%
    mutate(dataset = factor(dataset, levels = dataset_names)) %>%
    arrange(dataset) %>%
    mutate(dataset = as.character(dataset))

  missing_depths <- setdiff(dataset_names, depth_table$dataset)
  if (length(missing_depths) > 0) {
    stop(
      "No ", timepoint, " hpf total-cell counts found for: ",
      paste(missing_depths, collapse = ", "),
      ". Rerun the atlas comparison notebook combine/cache cells first."
    )
  }

  depth_table
}

cells_per_embryo_by_dataset <- derive_cells_per_embryo_by_dataset()

missing_cell_depths <- setdiff(datasets_to_compare, cells_per_embryo_by_dataset$dataset)
if (length(missing_cell_depths) > 0) {
  stop("Missing cells_per_embryo values for: ", paste(missing_cell_depths, collapse = ", "))
}

cells_per_embryo_slug <- cells_per_embryo_by_dataset %>%
  filter(dataset %in% datasets_to_compare) %>%
  mutate(dataset = factor(dataset, levels = datasets_to_compare)) %>%
  arrange(dataset) %>%
  mutate(
    dataset_slug = str_replace_all(dataset, "[^[:alnum:]]+", "_"),
    label = paste0(dataset_slug, "_", cells_per_embryo, "cells")
  ) %>%
  pull(label) %>%
  paste(collapse = "_")

simulation_model_slug <- "empirical_depth_multinomial_counts"
detection_model_slug <- "nominal_p_no_bh"
power_run_slug <- paste(cells_per_embryo_slug, simulation_model_slug, detection_model_slug, sep = "_")
other_cell_group <- "other cells"

effect_sizes <- seq(0.1, 0.9, by = 0.1)
embryo_counts <- c(8, 16, 24, 32)
n_sim <- 100
p_threshold <- 0.05
target_power <- 0.80
n_mde_bootstrap <- 500
base_seed <- 42

timepoint_col <- "timepoint"

# Keep FALSE until you actually want to launch the expensive NB GLM sweep.
run_power_sweep <- TRUE
force_rerun_power <- FALSE

target_slug <- str_replace_all(
  str_to_lower(paste(target_standardized_cell_types, collapse = "_")),
  "[^[:alnum:]]+",
  "_"
) %>% str_replace_all("^_|_$", "")

In [ ]:
cell_group_key <- tribble(
  ~dataset, ~cell_group, ~standardized_cell_group,
  "v2.3.0", "hypochord", "hypochord",
  "v2.3.0", "notochord (suspected doublets)", NA_character_,
  "v2.3.0", "notochordal cell", "notochordal cell",
  "v2.3.0", "notochordal cell (noto+)", "notochordal cell",
  "v2.3.0", "notochordal sheath cell (G2_M)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late, G2_M)", "notochordal sheath cell",
  "v2.3.0", "notochordal sheath cell (late, entpd5a+)", "notochordal sheath cell",
  "v2.3.0", "notochordal vacuole cell (early)", "notochordal vacuole cell",
  "v2.3.0", "notochordal vacuole cell (late)", "notochordal vacuole cell",
  "v3.1.0", "notochordal cell, notochord (11-18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal cell, notochord (18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal cell, notochord (posterior, 11-18 hpf)", "notochordal cell",
  "v3.1.0", "notochordal sheath cell, notochord", "notochordal sheath cell",
  "v3.1.0", "notochordal vacuole cell, notochord (18 hpf)", "notochordal vacuole cell",
  "v3.1.0", "notochordal vacuole cell, notochord (24-96 hpf)", "notochordal vacuole cell",
  "v3.1.0", "unknown, hypochord", "hypochord"
)

cell_group_key

In [ ]:
theme_power <- function(base_size = 18) {
  theme_bw(base_size = base_size) +
    theme(
      strip.background = element_rect(fill = "grey95"),
      strip.text = element_text(size = rel(0.85), lineheight = 0.95),
      panel.grid.minor = element_blank(),
      plot.title = element_text(size = rel(1.25), margin = margin(b = 6)),
      plot.subtitle = element_text(size = rel(0.85), margin = margin(b = 9), lineheight = 0.95),
      plot.margin = margin(t = 18, r = 24, b = 18, l = 24),
      axis.title = element_text(size = rel(0.85)),
      axis.text = element_text(size = rel(0.7)),
      axis.text.x = element_text(size = rel(0.65), angle = 45, hjust = 1),
      axis.title.y = element_text(margin = margin(r = 9)),
      axis.title.x = element_text(margin = margin(t = 9)),
      legend.text = element_text(size = rel(0.7), lineheight = 0.9),
      legend.title = element_text(size = rel(0.78)),
      legend.position = "bottom"
    )
}

save_power_plot <- function(plot, filename, width = 10, height = 7) {
  pdf_path <- file.path(power_out_dir, paste0(filename, ".pdf"))
  png_path <- file.path(power_out_dir, paste0(filename, ".png"))
  ggsave(pdf_path, plot, width = width, height = height, units = "in")
  ggsave(png_path, plot, width = width, height = height, units = "in", dpi = 300)
  message("Saved: ", pdf_path)
  message("Saved: ", png_path)
}

In [ ]:
sanitize_dataset_name <- function(dataset_name) {
  str_replace_all(dataset_name, "[^[:alnum:]_.-]+", "_")
}

dataset_cell_count_cache_path <- function(dataset_name) {
  file.path(count_cache_dir, paste0("embryo_cell_counts_", sanitize_dataset_name(dataset_name), ".rds"))
}

read_atlas_cell_counts <- function(dataset_name) {
  cache_path <- dataset_cell_count_cache_path(dataset_name)
  if (!file.exists(cache_path)) {
    stop("Missing cached embryo-level count table for ", dataset_name, ": ", cache_path)
  }

  readRDS(cache_path) %>%
    mutate(
      dataset = as.character(dataset),
      embryo = as.character(embryo),
      cell_group = as.character(cell_group)
    )
}

standardized_cell_group_set <- function() {
  groups <- cell_group_key %>%
    filter(!is.na(standardized_cell_group), standardized_cell_group != "") %>%
    distinct(cell_group = standardized_cell_group) %>%
    arrange(cell_group)

  if (!is.null(analysis_standardized_cell_groups)) {
    groups <- groups %>% filter(cell_group %in% analysis_standardized_cell_groups)
  }

  missing_targets <- setdiff(target_standardized_cell_types, groups$cell_group)
  if (length(missing_targets) > 0) {
    stop("Target cell type(s) are not in the analysis group set: ", paste(missing_targets, collapse = ", "))
  }

  groups
}

prepare_atlas_ccs <- function(dataset_name) {
  cells_per_embryo_i <- cells_per_embryo_by_dataset %>%
    filter(dataset == dataset_name) %>%
    pull(cells_per_embryo)
  if (length(cells_per_embryo_i) != 1 || is.na(cells_per_embryo_i)) {
    stop("Expected exactly one cells_per_embryo value for ", dataset_name)
  }

  raw_cell_counts <- read_atlas_cell_counts(dataset_name)
  raw_tp <- raw_cell_counts %>%
    filter(dataset == dataset_name, hpf == analysis_timepoint)

  embryo_total_cells <- raw_tp %>%
    group_by(dataset, embryo, timepoint = hpf) %>%
    summarise(total_recovered_cells = sum(cell_count, na.rm = TRUE), .groups = "drop") %>%
    mutate(
      dataset = as.character(dataset),
      embryo = as.character(embryo),
      timepoint = as.character(timepoint)
    )

  embryo_stage_grid <- embryo_total_cells %>%
    transmute(
      dataset = as.character(dataset),
      embryo = as.character(embryo),
      timepoint = as.character(timepoint)
    ) %>%
    distinct()

  if (nrow(embryo_stage_grid) == 0) {
    stop("No embryos found for ", dataset_name, " at ", analysis_timepoint, " hpf.")
  }

  analysis_groups <- standardized_cell_group_set()

  observed_counts <- raw_tp %>%
    inner_join(
      cell_group_key %>% filter(dataset == dataset_name, !is.na(standardized_cell_group)),
      by = c("dataset", "cell_group")
    ) %>%
    filter(standardized_cell_group %in% analysis_groups$cell_group) %>%
    group_by(dataset, embryo, timepoint = hpf, cell_group = standardized_cell_group) %>%
    summarise(cell_count = sum(cell_count, na.rm = TRUE), .groups = "drop") %>%
    mutate(timepoint = as.character(timepoint))

  count_long <- embryo_stage_grid %>%
    tidyr::crossing(analysis_groups) %>%
    left_join(
      observed_counts,
      by = c("dataset", "embryo", "timepoint", "cell_group")
    ) %>%
    left_join(
      embryo_total_cells,
      by = c("dataset", "embryo", "timepoint")
    ) %>%
    mutate(
      cell_count = tidyr::replace_na(cell_count, 0),
      cell_count_per_1000 = if_else(
        total_recovered_cells > 0,
        1000 * cell_count / total_recovered_cells,
        NA_real_
      ),
      dataset = factor(dataset, levels = datasets_to_compare),
      cell_group = factor(cell_group, levels = analysis_groups$cell_group)
    )

  other_counts <- count_long %>%
    group_by(dataset, embryo, timepoint) %>%
    summarise(
      modeled_cell_count = sum(cell_count, na.rm = TRUE),
      total_recovered_cells = first(total_recovered_cells),
      .groups = "drop"
    ) %>%
    mutate(
      cell_group = other_cell_group,
      cell_count = pmax(total_recovered_cells - modeled_cell_count, 0),
      cell_count_per_1000 = if_else(
        total_recovered_cells > 0,
        1000 * cell_count / total_recovered_cells,
        NA_real_
      )
    ) %>%
    select(dataset, embryo, timepoint, cell_group, cell_count, total_recovered_cells, cell_count_per_1000)

  model_count_long <- bind_rows(
    count_long %>%
      mutate(cell_group = as.character(cell_group)) %>%
      select(dataset, embryo, timepoint, cell_group, cell_count, total_recovered_cells, cell_count_per_1000),
    other_counts
  ) %>%
    mutate(
      dataset = factor(dataset, levels = datasets_to_compare),
      cell_group = factor(cell_group, levels = c(analysis_groups$cell_group, other_cell_group))
    )

  observed_count_wide <- model_count_long %>%
    select(cell_group, embryo, cell_count) %>%
    pivot_wider(names_from = embryo, values_from = cell_count, values_fill = 0) %>%
    column_to_rownames("cell_group") %>%
    as.matrix()

  observed_col_data <- embryo_total_cells %>%
    filter(embryo %in% colnames(observed_count_wide)) %>%
    arrange(match(embryo, colnames(observed_count_wide))) %>%
    transmute(
      embryo,
      dataset = dataset_name,
      timepoint = as.character(analysis_timepoint),
      total_recovered_cells,
      total_cells = as.numeric(colSums(observed_count_wide))
    ) %>%
    as.data.frame()
  rownames(observed_col_data) <- observed_col_data$embryo

  missing_targets <- setdiff(target_standardized_cell_types, rownames(observed_count_wide))
  if (length(missing_targets) > 0) {
    stop("Target cell type(s) missing from count matrix for ", dataset_name, ": ", paste(missing_targets, collapse = ", "))
  }

  ccs <- list(counts = observed_count_wide, col_data = observed_col_data)

  list(
    dataset = dataset_name,
    ccs = ccs,
    count_long = count_long,
    model_count_long = model_count_long,
    target_cell_types = target_standardized_cell_types,
    embryo_size = cells_per_embryo_i,
    observed_median_total_recovered_cells = round(median(observed_col_data$total_recovered_cells)),
    observed_sd_total_recovered_cells = sd(observed_col_data$total_recovered_cells),
    observed_min_total_recovered_cells = min(observed_col_data$total_recovered_cells),
    observed_max_total_recovered_cells = max(observed_col_data$total_recovered_cells)
  )
}

atlas_inputs <- setNames(lapply(datasets_to_compare, prepare_atlas_ccs), datasets_to_compare)
observed_counts_long <- bind_rows(lapply(atlas_inputs, `[[`, "count_long"))
model_counts_long <- bind_rows(lapply(atlas_inputs, `[[`, "model_count_long"))

atlas_input_summary <- purrr::imap_dfr(atlas_inputs, function(x, dataset_name) {
  tibble(
    dataset = dataset_name,
    n_embryos = ncol(x$ccs$counts),
    observed_median_total_recovered_cells = x$observed_median_total_recovered_cells,
    simulation_cells_per_embryo = x$embryo_size,
    observed_sd_total_recovered_cells = x$observed_sd_total_recovered_cells,
    observed_min_total_recovered_cells = x$observed_min_total_recovered_cells,
    observed_max_total_recovered_cells = x$observed_max_total_recovered_cells,
    target_cell_types = paste(x$target_cell_types, collapse = ", ")
  )
})

write.csv(
  cells_per_embryo_by_dataset,
  file.path(power_out_dir, sprintf("cells_per_embryo_by_dataset_%s_%shpf.csv", target_slug, analysis_timepoint)),
  row.names = FALSE
)
write.csv(
  observed_counts_long,
  file.path(power_out_dir, sprintf("atlas_power_observed_counts_%s_%shpf.csv", target_slug, analysis_timepoint)),
  row.names = FALSE
)
write.csv(
  model_counts_long,
  file.path(power_out_dir, sprintf("atlas_power_model_counts_full_composition_%s_%shpf.csv", target_slug, analysis_timepoint)),
  row.names = FALSE
)

atlas_input_summary

In [ ]:
observed_target_counts <- observed_counts_long %>%
  filter(as.character(cell_group) %in% target_standardized_cell_types) %>%
  left_join(
    cells_per_embryo_by_dataset %>% select(dataset, cells_per_embryo),
    by = "dataset"
  ) %>%
  mutate(expected_cells_per_embryo = cell_count_per_1000 * cells_per_embryo / 1000)

p_observed_counts <- ggplot(
  observed_target_counts,
  aes(x = dataset, y = cell_count_per_1000, fill = dataset, color = dataset)
) +
  geom_boxplot(outlier.shape = NA, alpha = 0.60, width = 0.55, color = "grey25") +
  geom_jitter(width = 0.12, alpha = 0.55, size = 1.4) +
  facet_wrap(~ cell_group, scales = "free_y", labeller = labeller(cell_group = label_wrap_gen(width = 18))) +
  scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 14)) +
  scale_fill_brewer(palette = "Set2", drop = FALSE) +
  scale_color_brewer(palette = "Dark2", drop = FALSE) +
  labs(
    title = "Observed harmonized sheath-cell abundance by atlas",
    subtitle = str_wrap(sprintf("%s hpf; one point per embryo", analysis_timepoint), width = 72),
    x = "Atlas",
    y = "Cells per 1,000\nrecovered cells"
  ) +
  theme_power(base_size = 20) +
  theme(legend.position = "none")

save_power_plot(
  p_observed_counts,
  sprintf("observed_counts_per_1000_%s_%shpf_by_atlas", target_slug, analysis_timepoint),
  width = 9,
  height = 6
)

p_observed_counts

p_observed_expected_counts <- ggplot(
  observed_target_counts,
  aes(x = dataset, y = expected_cells_per_embryo, fill = dataset, color = dataset)
) +
  geom_boxplot(outlier.shape = NA, alpha = 0.60, width = 0.55, color = "grey25") +
  geom_jitter(width = 0.12, alpha = 0.55, size = 1.4) +
  facet_wrap(~ cell_group, scales = "free_y", labeller = labeller(cell_group = label_wrap_gen(width = 18))) +
  scale_x_discrete(labels = function(x) stringr::str_wrap(x, width = 14)) +
  scale_fill_brewer(palette = "Set2", drop = FALSE) +
  scale_color_brewer(palette = "Dark2", drop = FALSE) +
  labs(
    title = "Expected sheath cells per embryo by atlas",
    subtitle = str_wrap(sprintf("%s hpf; per-1,000 abundance rescaled by dataset-specific median recovered cells", analysis_timepoint), width = 72),
    x = "Atlas",
    y = "Expected cells\nper embryo"
  ) +
  theme_power(base_size = 20) +
  theme(legend.position = "none")

save_power_plot(
  p_observed_expected_counts,
  sprintf("observed_expected_cells_per_embryo_%s_%shpf_by_atlas", target_slug, analysis_timepoint),
  width = 9,
  height = 6
)

p_observed_expected_counts

## Simulation And NB GLM Functions

These are copied from the active plain R workflow in `power_test_notebook.ipynb` and parameterized so the same sweep can run independently for each atlas.

In [ ]:
get_cell_count_wide <- function(ccs) {
  ccs$counts %>% as.matrix() %>% as.data.frame()
}

get_timepoint_levels <- function(ccs, timepoint_col = "timepoint", timepoints = NULL) {
  md <- as.data.frame(ccs$col_data)
  available <- sort(unique(as.character(md[[timepoint_col]])))
  available <- available[!is.na(available) & nzchar(available)]
  if (is.null(timepoints)) return(available)

  missing <- setdiff(timepoints, available)
  if (length(missing) > 0) {
    stop(sprintf("Timepoint(s) not found: %s", paste(missing, collapse = ", ")))
  }

  available[available %in% timepoints]
}

subset_ccs_by_timepoint <- function(ccs, timepoint_value, timepoint_col = "timepoint") {
  md <- as.data.frame(ccs$col_data)
  keep <- rownames(md)[as.character(md[[timepoint_col]]) == as.character(timepoint_value)]
  list(
    counts = ccs$counts[, keep, drop = FALSE],
    col_data = md[keep, , drop = FALSE]
  )
}

get_prop_mat <- function(cell_count_wide, pseudocount = 1) {
  cell_count_wide %>%
    tibble::rownames_to_column("cell_group") %>%
    tidyr::pivot_longer(-cell_group, names_to = "embryo", values_to = "count") %>%
    dplyr::mutate(count = count + pseudocount) %>%
    dplyr::group_by(embryo) %>%
    dplyr::mutate(prop = count / sum(count)) %>%
    dplyr::select(-count) %>%
    tidyr::pivot_wider(names_from = cell_group, values_from = prop) %>%
    tibble::column_to_rownames("embryo") %>%
    as.matrix()
}

fit_drichlet <- function(prop_mat, model_formula_str = "~ 1", trace = FALSE) {
  model_formula <- as.formula(paste("prop_mat", model_formula_str))
  VGAM::vglm(
    model_formula,
    data = as.data.frame(prop_mat),
    family = "dirichlet",
    trace = trace
  )
}

sample_total_cells <- function(ccs, num_embryos, embryo_size = 1000) {
  depth_pool <- NULL
  if (!is.null(ccs$col_data) && "total_recovered_cells" %in% colnames(ccs$col_data)) {
    depth_pool <- as.numeric(ccs$col_data$total_recovered_cells)
  } else if (!is.null(ccs$col_data) && "total_cells" %in% colnames(ccs$col_data)) {
    depth_pool <- as.numeric(ccs$col_data$total_cells)
  }

  if (is.null(depth_pool)) depth_pool <- numeric()
  depth_pool <- as.integer(round(depth_pool[is.finite(depth_pool) & depth_pool > 0]))
  if (length(depth_pool) == 0) {
    return(pmax(1L, as.integer(stats::rpois(n = num_embryos, lambda = embryo_size))))
  }

  sample(depth_pool, size = num_embryos, replace = TRUE)
}

sample_count_matrix <- function(prop_wide, total_cells) {
  prop_mat <- as.matrix(prop_wide[, setdiff(colnames(prop_wide), "embryo"), drop = FALSE])
  storage.mode(prop_mat) <- "numeric"

  count_mat <- vapply(seq_len(nrow(prop_mat)), function(i) {
    probs <- pmax(prop_mat[i, ], 0)
    prob_sum <- sum(probs)
    if (!is.finite(prob_sum) || prob_sum <= 0) {
      probs <- rep(1 / ncol(prop_mat), ncol(prop_mat))
    } else {
      probs <- probs / prob_sum
    }
    as.integer(stats::rmultinom(1, size = total_cells[[i]], prob = probs)[, 1])
  }, integer(ncol(prop_mat)))

  rownames(count_mat) <- colnames(prop_mat)
  colnames(count_mat) <- seq_len(nrow(prop_mat))
  count_mat
}

simulate_single_timepoint <- function(ccs, embryo_size = 1000, random.seed = 111,
                                      genotype = "WT", num_embryos = NULL,
                                      timepoint_value = NULL) {
  cell_count_wide <- get_cell_count_wide(ccs)
  prop_mat <- get_prop_mat(cell_count_wide)
  dfit <- fit_drichlet(prop_mat)

  if (is.null(num_embryos)) num_embryos <- ncol(cell_count_wide)
  num_sims <- ceiling(num_embryos / ncol(cell_count_wide))

  n_obs <- nrow(prop_mat)
  n_types <- ncol(prop_mat)
  dsim <- VGAM::simulate.vlm(dfit, nsim = num_sims, seed = random.seed)
  if (!is.list(dsim)) dsim <- list(dsim)
  dsim <- lapply(dsim, function(d) {
    d <- matrix(d, nrow = n_obs, ncol = n_types, byrow = FALSE)
    colnames(d) <- colnames(prop_mat)
    d
  })

  sim_prop_wide <- dplyr::bind_rows(lapply(dsim, as.data.frame))
  sim_prop_wide <- sim_prop_wide[seq_len(num_embryos), , drop = FALSE]
  sim_prop_wide$embryo <- seq_len(nrow(sim_prop_wide))

  set.seed(random.seed + 17)
  total_cells <- sample_total_cells(ccs, num_embryos = num_embryos, embryo_size = embryo_size)
  sim_count_wide <- sample_count_matrix(sim_prop_wide, total_cells = total_cells)

  tp_tag <- if (is.null(timepoint_value)) "all" else as.character(timepoint_value)
  colnames(sim_count_wide) <- paste0("embryo-", genotype, "-", tp_tag, "-", seq_len(ncol(sim_count_wide)))

  col_data <- data.frame(
    embryo = colnames(sim_count_wide),
    genotype = genotype,
    timepoint = tp_tag,
    embryo_size = embryo_size,
    total_cells = as.integer(total_cells),
    stringsAsFactors = FALSE
  )
  rownames(col_data) <- col_data$embryo

  list(counts = sim_count_wide, col_data = col_data)
}

simulate_embryos <- function(ccs, embryo_size = 1000, random.seed = 111,
                             genotype = "WT", num_embryos = 10,
                             timepoints = NULL, timepoint_col = "timepoint") {
  selected_timepoints <- get_timepoint_levels(ccs, timepoint_col, timepoints)

  parts <- lapply(seq_along(selected_timepoints), function(i) {
    tp <- selected_timepoints[[i]]
    suppressWarnings(
      simulate_single_timepoint(
        subset_ccs_by_timepoint(ccs, tp, timepoint_col),
        embryo_size = embryo_size,
        random.seed = random.seed + i * 1000,
        genotype = genotype,
        num_embryos = num_embryos,
        timepoint_value = tp
      )
    )
  })

  list(
    counts = do.call(cbind, lapply(parts, `[[`, "counts")),
    col_data = dplyr::bind_rows(lapply(parts, `[[`, "col_data"))
  )
}

mutate_counts <- function(sim, cell_types, effect_size, sink_cell_type = other_cell_group,
                          random.seed = NULL) {
  if (!is.null(random.seed)) set.seed(random.seed)
  mat <- sim$counts
  removed_counts <- rep(0, ncol(mat))
  for (ct in cell_types) {
    if (ct %in% rownames(mat)) {
      old_count <- mat[ct, ]
      new_count <- stats::rbinom(length(old_count), size = as.integer(old_count), prob = 1 - effect_size)
      mat[ct, ] <- new_count
      removed_counts <- removed_counts + pmax(old_count - new_count, 0)
    }
  }
  if (sink_cell_type %in% rownames(mat)) {
    mat[sink_cell_type, ] <- mat[sink_cell_type, ] + removed_counts
  }
  sim$counts <- mat
  sim$col_data$mutated <- TRUE
  sim$col_data$effect_size <- effect_size
  sim
}

combine_sims <- function(sim_list) {
  list(
    counts = do.call(cbind, lapply(sim_list, `[[`, "counts")),
    col_data = dplyr::bind_rows(lapply(sim_list, `[[`, "col_data"))
  )
}

In [ ]:
extract_timepoint_effects <- function(fit, timepoint_levels, timepoint_col) {
  coefs <- coef(fit)
  vcov_m <- vcov(fit)
  geno <- "genotypeMT"

  dplyr::bind_rows(lapply(timepoint_levels, function(tp) {
    int_term <- intersect(
      c(
        paste0("genotypeMT:", timepoint_col, tp),
        paste0(timepoint_col, tp, ":genotypeMT")
      ),
      names(coefs)
    )
    terms_used <- c(geno, int_term)

    if (!all(terms_used %in% names(coefs))) {
      return(tibble::tibble(
        timepoint = tp,
        estimated_effect = NA_real_,
        std_error = NA_real_,
        p_value = NA_real_
      ))
    }

    est <- sum(coefs[terms_used])
    se <- sqrt(sum(vcov_m[terms_used, terms_used]))

    tibble::tibble(
      timepoint = tp,
      estimated_effect = unname(est),
      std_error = unname(se),
      p_value = 2 * pnorm(abs(est / se), lower.tail = FALSE)
    )
  }))
}

run_trial_nb <- function(ccs, target_cell_types, effect_size, num_embryos,
                         embryo_size, seed, p_threshold,
                         timepoints = NULL, timepoint_col = "timepoint") {
  selected_timepoints <- get_timepoint_levels(ccs, timepoint_col, timepoints)

  wt <- simulate_embryos(
    ccs,
    embryo_size = embryo_size,
    random.seed = seed,
    genotype = "WT",
    num_embryos = num_embryos,
    timepoints = selected_timepoints,
    timepoint_col = timepoint_col
  )

  mt <- simulate_embryos(
    ccs,
    embryo_size = embryo_size,
    random.seed = seed + 1e6,
    genotype = "MT",
    num_embryos = num_embryos,
    timepoints = selected_timepoints,
    timepoint_col = timepoint_col
  )
  mt <- mutate_counts(mt, target_cell_types, effect_size, random.seed = seed + 2e6)

  combined <- combine_sims(list(wt, mt))
  count_mat <- combined$counts
  col_data <- combined$col_data
  col_data$log_total <- log(pmax(colSums(count_mat), 1))
  col_data$genotype <- factor(col_data$genotype, levels = c("WT", "MT"))
  col_data[[timepoint_col]] <- factor(col_data$timepoint)

  if (length(selected_timepoints) == 1) {
    fml <- count ~ genotype + offset(log_total)
  } else {
    fml <- as.formula(sprintf("count ~ genotype * %s + offset(log_total)", timepoint_col))
  }

  result <- dplyr::bind_rows(lapply(rownames(count_mat), function(ct) {
    col_data$count <- as.numeric(count_mat[ct, ])
    fit <- tryCatch(
      suppressWarnings(MASS::glm.nb(fml, data = col_data)),
      error = function(e) NULL
    )

    if (is.null(fit)) {
      return(tidyr::expand_grid(timepoint = selected_timepoints) %>%
        dplyr::mutate(
          cell_group = ct,
          estimated_effect = NA_real_,
          std_error = NA_real_,
          p_value = NA_real_,
          model_converged = FALSE
        ))
    }

    extract_timepoint_effects(fit, selected_timepoints, timepoint_col) %>%
      dplyr::mutate(cell_group = ct, model_converged = TRUE)
  }))

  result %>%
    dplyr::mutate(
      is_target = cell_group %in% target_cell_types,
      true_effect = ifelse(is_target, log(1 - effect_size), 0)
    ) %>%
    dplyr::mutate(
      detection_p_value = p_value,
      detected = ifelse(!is.na(detection_p_value), detection_p_value < p_threshold, NA),
      ci_lower = estimated_effect - 1.96 * std_error,
      ci_upper = estimated_effect + 1.96 * std_error,
      ci_covers_truth = ifelse(
        !is.na(ci_lower) & !is.na(ci_upper),
        ci_lower <= true_effect & true_effect <= ci_upper,
        NA
      )
    )
}

In [ ]:
power_results_path <- file.path(
  power_out_dir,
  sprintf("atlas_power_results_nb_full_composition_%s_%shpf_%s.rds", target_slug, analysis_timepoint, power_run_slug)
)
power_results_csv_path <- str_replace(power_results_path, "\\.rds$", ".csv")

run_power_sweep_for_atlas <- function(atlas_input, atlas_index = 1) {
  grid <- expand.grid(
    effect_size = effect_sizes,
    num_embryos = embryo_counts,
    sim_idx = seq_len(n_sim),
    stringsAsFactors = FALSE
  )

  cat(sprintf(
    "Running %d NB GLM trials for %s...
",
    nrow(grid),
    atlas_input$dataset
  ))
  start_time <- Sys.time()

  results <- dplyr::bind_rows(lapply(seq_len(nrow(grid)), function(i) {
    if (i %% 20 == 0) {
      cat(sprintf(
        "  %s: %d/%d (%.1f min)
",
        atlas_input$dataset,
        i,
        nrow(grid),
        as.numeric(difftime(Sys.time(), start_time, units = "mins"))
      ))
    }

    run_trial_nb(
      ccs = atlas_input$ccs,
      target_cell_types = atlas_input$target_cell_types,
      effect_size = grid$effect_size[[i]],
      num_embryos = grid$num_embryos[[i]],
      embryo_size = atlas_input$embryo_size,
      seed = base_seed + atlas_index * 100000 + i * 7,
      p_threshold = p_threshold,
      timepoints = as.character(analysis_timepoint),
      timepoint_col = timepoint_col
    ) %>%
      dplyr::mutate(
        dataset = atlas_input$dataset,
        effect_size_param = grid$effect_size[[i]],
        num_embryos_param = grid$num_embryos[[i]],
        sim_idx = grid$sim_idx[[i]],
        embryo_size_param = atlas_input$embryo_size,
        simulation_cells_per_embryo = atlas_input$embryo_size,
        observed_median_total_recovered_cells = atlas_input$observed_median_total_recovered_cells,
        observed_sd_total_recovered_cells = atlas_input$observed_sd_total_recovered_cells,
        simulation_depth_model = "sample observed total recovered-cell depths with replacement",
        count_sampling_model = "dirichlet proportions plus multinomial recovered-cell sampling; target binomial thinning",
        simulation_model_slug = simulation_model_slug,
        detection_model_slug = detection_model_slug,
        detection_threshold = p_threshold,
        model_denominator = "full recovered cells plus other-cell complement",
        n_observed_embryos = ncol(atlas_input$ccs$counts)
      )
  }))

  cat(sprintf(
    "Done with %s. %.1f min total.
",
    atlas_input$dataset,
    as.numeric(difftime(Sys.time(), start_time, units = "mins"))
  ))

  results
}

if (file.exists(power_results_path) && !force_rerun_power) {
  message("Reading cached power results: ", power_results_path)
  all_results_nb <- readRDS(power_results_path)
} else if (isTRUE(run_power_sweep)) {
  all_results_nb <- purrr::imap_dfr(
    atlas_inputs,
    function(atlas_input, dataset_name) {
      atlas_index <- match(dataset_name, datasets_to_compare)
      run_power_sweep_for_atlas(atlas_input, atlas_index = atlas_index)
    }
  )

  saveRDS(all_results_nb, power_results_path)
  write.csv(all_results_nb, power_results_csv_path, row.names = FALSE)
  message("Saved: ", power_results_path)
  message("Saved: ", power_results_csv_path)
} else {
  all_results_nb <- NULL
  message("No cached power results were loaded.")
  message("Set run_power_sweep <- TRUE, then rerun this cell to launch simulations.")
}

if (!is.null(all_results_nb)) {
  dplyr::glimpse(all_results_nb)
}

## Summaries

The MDE estimate is interpolated between simulated fractional reductions whose estimated power brackets `target_power`. Points are bootstrap mean interpolated MDEs; error bars are mean ± bootstrap SE over simulation replicates.

In [ ]:
require_power_results <- function() {
  if (!exists("all_results_nb") || is.null(all_results_nb)) {
    stop("No power results are available. Load cached results or set run_power_sweep <- TRUE and rerun the sweep cell.")
  }
  if (!"detection_p_value" %in% colnames(all_results_nb)) {
    stop("Power results do not contain nominal endpoint p-values. Rerun the sweep with the no-BH detection model.")
  }
  if ("detection_model_slug" %in% colnames(all_results_nb) &&
      any(all_results_nb$detection_model_slug != detection_model_slug, na.rm = TRUE)) {
    stop("Power results were generated with a different detection model. Rerun the sweep for ", detection_model_slug, ".")
  }
}

summarise_power <- function(all_results) {
  all_results %>%
    filter(is_target, model_converged) %>%
    mutate(
      effect_size_param = as.numeric(as.character(effect_size_param)),
      num_embryos_param = as.numeric(as.character(num_embryos_param))
    ) %>%
    group_by(dataset, cell_group, effect_size_param, num_embryos_param, timepoint) %>%
    summarise(
      power = mean(detected, na.rm = TRUE),
      n_trials = sum(!is.na(detected)),
      power_se = sqrt(power * (1 - power) / pmax(n_trials, 1)),
      power_lwr = pmax(0, power - 1.96 * power_se),
      power_upr = pmin(1, power + 1.96 * power_se),
      median_p_value = median(detection_p_value, na.rm = TRUE),
      .groups = "drop"
    )
}

interpolate_mde <- function(effect_size, power, target_power = 0.80) {
  power_df <- tibble(effect_size = effect_size, power = power) %>%
    filter(!is.na(effect_size), !is.na(power)) %>%
    arrange(effect_size) %>%
    group_by(effect_size) %>%
    summarise(power = mean(power, na.rm = TRUE), .groups = "drop")

  if (nrow(power_df) == 0 || max(power_df$power, na.rm = TRUE) < target_power) {
    return(NA_real_)
  }

  power_df <- power_df %>%
    mutate(power_monotone = cummax(power))

  hit_idx <- which(power_df$power_monotone >= target_power)[[1]]
  if (hit_idx == 1) {
    return(power_df$effect_size[[1]])
  }

  lo <- power_df[hit_idx - 1, ]
  hi <- power_df[hit_idx, ]
  if (hi$power_monotone == lo$power_monotone) {
    return(hi$effect_size)
  }

  lo$effect_size +
    (target_power - lo$power_monotone) /
    (hi$power_monotone - lo$power_monotone) *
    (hi$effect_size - lo$effect_size)
}

mde_from_power <- function(power_df, target_power = 0.80) {
  tibble(
    mde_mean_power = interpolate_mde(
      effect_size = power_df$effect_size_param,
      power = power_df$power,
      target_power = target_power
    )
  )
}

bootstrap_mde <- function(all_results, target_power = 0.80, n_boot = 500) {
  all_results %>%
    filter(is_target, model_converged, !is.na(detected)) %>%
    mutate(
      effect_size_param = as.numeric(as.character(effect_size_param)),
      num_embryos_param = as.numeric(as.character(num_embryos_param))
    ) %>%
    group_by(dataset, cell_group, num_embryos_param, timepoint) %>%
    group_modify(function(.x, .y) {
      effects <- sort(unique(.x$effect_size_param))

      boot_vals <- replicate(n_boot, {
        boot_power <- purrr::map_dbl(effects, function(effect_size_i) {
          vals <- .x$detected[.x$effect_size_param == effect_size_i]
          mean(sample(vals, length(vals), replace = TRUE), na.rm = TRUE)
        })

        interpolate_mde(
          effect_size = effects,
          power = boot_power,
          target_power = target_power
        )
      })

      n_detectable <- sum(!is.na(boot_vals))
      mde_boot_mean <- if (n_detectable == 0) NA_real_ else mean(boot_vals, na.rm = TRUE)
      mde_boot_sd <- if (n_detectable <= 1) NA_real_ else sd(boot_vals, na.rm = TRUE)
      mde_boot_se <- mde_boot_sd
      mde_boot_mcse <- if (n_detectable <= 1) NA_real_ else mde_boot_sd / sqrt(n_detectable)

      tibble(
        mde_boot_mean = mde_boot_mean,
        mde_boot_sd = mde_boot_sd,
        mde_boot_se = mde_boot_se,
        mde_boot_mcse = mde_boot_mcse,
        mde_lwr = if (is.na(mde_boot_se)) NA_real_ else pmax(0, mde_boot_mean - mde_boot_se),
        mde_upr = if (is.na(mde_boot_se)) NA_real_ else pmin(1, mde_boot_mean + mde_boot_se),
        n_boot_detectable = n_detectable
      )
    }) %>%
    ungroup()
}

require_power_results()

power_summary <- summarise_power(all_results_nb)

mde_point <- power_summary %>%
  group_by(dataset, cell_group, num_embryos_param, timepoint) %>%
  group_modify(~ mde_from_power(.x, target_power = target_power)) %>%
  ungroup()

mde_summary <- mde_point %>%
  left_join(
    bootstrap_mde(all_results_nb, target_power = target_power, n_boot = n_mde_bootstrap),
    by = c("dataset", "cell_group", "num_embryos_param", "timepoint")
  ) %>%
  mutate(mde = mde_boot_mean)

write.csv(
  power_summary,
  file.path(power_out_dir, sprintf("atlas_power_summary_full_composition_%s_%shpf_%s.csv", target_slug, analysis_timepoint, power_run_slug)),
  row.names = FALSE
)
write.csv(
  mde_summary,
  file.path(power_out_dir, sprintf("atlas_power_mde_summary_full_composition_%s_%shpf_%s.csv", target_slug, analysis_timepoint, power_run_slug)),
  row.names = FALSE
)

power_summary
mde_summary

In [ ]:
require_power_results()

calibration_df <- all_results_nb %>%
  filter(is_target, model_converged, !is.na(estimated_effect)) %>%
  group_by(dataset, cell_group, effect_size_param, num_embryos_param, timepoint) %>%
  summarise(
    true_effect = first(true_effect),
    mean_estimated = mean(estimated_effect, na.rm = TRUE),
    se_estimated = sd(estimated_effect, na.rm = TRUE) / sqrt(n()),
    n_trials = n(),
    .groups = "drop"
  )

effect_plot_floor <- -5.0

calibration_plot_df <- calibration_df %>%
  mutate(
    mean_estimated_plot = pmax(mean_estimated, effect_plot_floor),
    estimated_lwr_plot = pmax(mean_estimated - 1.96 * se_estimated, effect_plot_floor),
    estimated_upr_plot = pmax(mean_estimated + 1.96 * se_estimated, effect_plot_floor)
  )

effect_range <- range(
  c(calibration_plot_df$true_effect, calibration_plot_df$mean_estimated_plot, effect_plot_floor),
  na.rm = TRUE
)

p_calibration <- ggplot(
  calibration_plot_df,
  aes(
    x = true_effect,
    y = mean_estimated_plot,
    color = factor(num_embryos_param),
    group = factor(num_embryos_param)
  )
) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey45") +
  geom_errorbar(aes(ymin = estimated_lwr_plot, ymax = estimated_upr_plot), width = 0) +
  geom_line(alpha = 0.55) +
  geom_point(size = 2.3) +
  facet_grid(cell_group ~ dataset, labeller = labeller(cell_group = label_wrap_gen(width = 18), dataset = label_wrap_gen(width = 16))) +
  coord_equal(xlim = effect_range, ylim = effect_range) +
  labs(
    title = "Effect size calibration by atlas",
    subtitle = str_wrap(sprintf("%s hpf; full-composition simulation with dataset-specific cells per embryo", analysis_timepoint), width = 80),
    x = "True effect\n(log scale)",
    y = "Mean estimated effect\n(log scale; capped at -5)",
    color = "Embryos per arm"
  ) +
  theme_power(base_size = 18)

save_power_plot(
  p_calibration,
  sprintf("effect_size_calibration_full_composition_%s_%shpf_%s_by_atlas", target_slug, analysis_timepoint, power_run_slug),
  width = 14,
  height = 8
)

p_calibration

In [ ]:
require_power_results()

pvalue_df <- all_results_nb %>%
  filter(is_target, model_converged, !is.na(detection_p_value)) %>%
  mutate(
    effect_size_param = factor(effect_size_param, levels = effect_sizes),
    num_embryos_param = factor(num_embryos_param, levels = embryo_counts),
    p_value_plot = pmax(detection_p_value, .Machine$double.xmin)
  )

p_pvalues <- ggplot(
  pvalue_df,
  aes(x = num_embryos_param, y = p_value_plot, fill = effect_size_param)
) +
  geom_boxplot(outlier.alpha = 0.25, position = position_dodge2(width = 0.75, preserve = "single")) +
  geom_hline(yintercept = p_threshold, linetype = "dashed", color = "grey35") +
  facet_grid(cell_group ~ dataset, labeller = labeller(cell_group = label_wrap_gen(width = 18), dataset = label_wrap_gen(width = 16))) +
  scale_y_log10(limits = c(1e-10, 1), oob = scales::squish) +
  scale_fill_brewer(palette = "Set2", drop = FALSE) +
  guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = "Target-cell p-values by atlas",
    subtitle = str_wrap(sprintf("%s hpf; pre-specified endpoint; dashed line = nominal p < %.2f", analysis_timepoint, p_threshold), width = 80),
    x = "Embryos per arm",
    y = "Nominal\np-value",
    fill = "Fractional reduction"
  ) +
  theme_power(base_size = 18)

save_power_plot(
  p_pvalues,
  sprintf("pvalue_distributions_full_composition_%s_%shpf_%s_by_atlas", target_slug, analysis_timepoint, power_run_slug),
  width = 14,
  height = 8
)

p_pvalues

pvalue_trend_df <- pvalue_df %>%
  group_by(dataset, cell_group, effect_size_param, num_embryos_param) %>%
  summarise(
    median_p_value = median(detection_p_value, na.rm = TRUE),
    p_value_lwr = quantile(detection_p_value, 0.25, na.rm = TRUE),
    p_value_upr = quantile(detection_p_value, 0.75, na.rm = TRUE),
    n_trials = n(),
    .groups = "drop"
  ) %>%
  mutate(
    num_embryos_param = as.numeric(as.character(num_embryos_param)),
    median_p_value_plot = pmax(median_p_value, 1e-10),
    p_value_lwr_plot = pmax(p_value_lwr, 1e-10),
    p_value_upr_plot = pmax(p_value_upr, 1e-10)
  )

p_pvalue_trends <- ggplot(
  pvalue_trend_df,
  aes(
    x = num_embryos_param,
    y = median_p_value_plot,
    color = effect_size_param,
    group = effect_size_param
  )
) +
  geom_hline(yintercept = p_threshold, linetype = "dashed", color = "grey35") +
  geom_errorbar(aes(ymin = p_value_lwr_plot, ymax = p_value_upr_plot), width = 0.8, alpha = 0.55) +
  geom_line(linewidth = 0.9) +
  geom_point(size = 2.4) +
  facet_grid(cell_group ~ dataset, labeller = labeller(cell_group = label_wrap_gen(width = 18), dataset = label_wrap_gen(width = 16))) +
  scale_x_continuous(breaks = embryo_counts) +
  scale_y_log10(limits = c(1e-10, 1), oob = scales::squish) +
  scale_color_brewer(palette = "Set2", drop = FALSE) +
  guides(color = guide_legend(nrow = 2, byrow = TRUE)) +
  labs(
    title = "Target-cell p-value trends by atlas",
    subtitle = sprintf("%s hpf; median with IQR; dashed = p < %.2f", analysis_timepoint, p_threshold),
    x = "Embryos per arm",
    y = "Median nominal\np-value",
    color = "Fractional reduction"
  ) +
  theme_power(base_size = 18) +
  theme(axis.text.x = element_text(size = 18, angle = 45, hjust = 1))

save_power_plot(
  p_pvalue_trends,
  sprintf("pvalue_trends_full_composition_%s_%shpf_%s_by_atlas", target_slug, analysis_timepoint, power_run_slug),
  width = 14,
  height = 8
)

p_pvalue_trends

In [ ]:
require_power_results()

summarise_power_for_mde_plot <- function(all_results) {
  all_results %>%
    filter(is_target, model_converged) %>%
    mutate(
      effect_size_param = as.numeric(as.character(effect_size_param)),
      num_embryos_param = as.numeric(as.character(num_embryos_param))
    ) %>%
    group_by(dataset, cell_group, effect_size_param, num_embryos_param, timepoint) %>%
    summarise(
      power = mean(detected, na.rm = TRUE),
      .groups = "drop"
    )
}

interpolate_mde_for_plot <- function(effect_size, power, target_power = 0.80) {
  power_df <- tibble(effect_size = effect_size, power = power) %>%
    filter(!is.na(effect_size), !is.na(power)) %>%
    arrange(effect_size) %>%
    group_by(effect_size) %>%
    summarise(power = mean(power, na.rm = TRUE), .groups = "drop")

  if (nrow(power_df) == 0 || max(power_df$power, na.rm = TRUE) < target_power) {
    return(NA_real_)
  }

  power_df <- power_df %>%
    mutate(power_monotone = cummax(power))

  hit_idx <- which(power_df$power_monotone >= target_power)[[1]]
  if (hit_idx == 1) {
    return(power_df$effect_size[[1]])
  }

  lo <- power_df[hit_idx - 1, ]
  hi <- power_df[hit_idx, ]
  if (hi$power_monotone == lo$power_monotone) {
    return(hi$effect_size)
  }

  lo$effect_size +
    (target_power - lo$power_monotone) /
    (hi$power_monotone - lo$power_monotone) *
    (hi$effect_size - lo$effect_size)
}

mde_from_power_for_plot <- function(power_df, target_power = 0.80) {
  tibble(
    mde_mean_power = interpolate_mde_for_plot(
      effect_size = power_df$effect_size_param,
      power = power_df$power,
      target_power = target_power
    )
  )
}

bootstrap_mde_mean_se_for_plot <- function(all_results, target_power = 0.80, n_boot = 500) {
  all_results %>%
    filter(is_target, model_converged, !is.na(detected)) %>%
    mutate(
      effect_size_param = as.numeric(as.character(effect_size_param)),
      num_embryos_param = as.numeric(as.character(num_embryos_param))
    ) %>%
    group_by(dataset, cell_group, num_embryos_param, timepoint) %>%
    group_modify(function(.x, .y) {
      effects <- sort(unique(.x$effect_size_param))

      boot_vals <- replicate(n_boot, {
        boot_power <- purrr::map_dbl(effects, function(effect_size_i) {
          vals <- .x$detected[.x$effect_size_param == effect_size_i]
          mean(sample(vals, length(vals), replace = TRUE), na.rm = TRUE)
        })

        interpolate_mde_for_plot(
          effect_size = effects,
          power = boot_power,
          target_power = target_power
        )
      })

      n_detectable <- sum(!is.na(boot_vals))
      mde_boot_mean <- if (n_detectable == 0) NA_real_ else mean(boot_vals, na.rm = TRUE)
      mde_boot_sd <- if (n_detectable <= 1) NA_real_ else sd(boot_vals, na.rm = TRUE)
      mde_boot_se <- mde_boot_sd
      mde_boot_mcse <- if (n_detectable <= 1) NA_real_ else mde_boot_sd / sqrt(n_detectable)

      tibble(
        mde_boot_mean = mde_boot_mean,
        mde_boot_sd = mde_boot_sd,
        mde_boot_se = mde_boot_se,
        mde_boot_mcse = mde_boot_mcse,
        n_boot_detectable = n_detectable
      )
    }) %>%
    ungroup()
}

mde_plot_power_summary <- summarise_power_for_mde_plot(all_results_nb)

mde_plot_point <- mde_plot_power_summary %>%
  group_by(dataset, cell_group, num_embryos_param, timepoint) %>%
  group_modify(~ mde_from_power_for_plot(.x, target_power = target_power)) %>%
  ungroup()

mde_summary <- mde_plot_point %>%
  left_join(
    bootstrap_mde_mean_se_for_plot(all_results_nb, target_power = target_power, n_boot = n_mde_bootstrap),
    by = c("dataset", "cell_group", "num_embryos_param", "timepoint")
  ) %>%
  mutate(
    mde = mde_boot_mean,
    mde_lwr = if_else(!is.na(mde_boot_se), pmax(0, mde - mde_boot_se), NA_real_),
    mde_upr = if_else(!is.na(mde_boot_se), pmin(1, mde + mde_boot_se), NA_real_)
  )

p_mde <- ggplot(
  mde_summary,
  aes(x = num_embryos_param, y = mde, color = dataset, group = dataset)
) +
  geom_errorbar(aes(ymin = mde_lwr, ymax = mde_upr), width = 0.8, linewidth = 0.7, na.rm = TRUE) +
  geom_line(linewidth = 0.9, na.rm = TRUE) +
  geom_point(size = 2.8, na.rm = TRUE) +
  facet_wrap(~ cell_group, labeller = labeller(cell_group = label_wrap_gen(width = 18))) +
  scale_x_continuous(breaks = embryo_counts) +
  scale_y_continuous(
    limits = c(0, 1),
    breaks = effect_sizes,
    labels = scales::percent_format(accuracy = 1)
  ) +
  scale_color_brewer(palette = "Dark2", drop = FALSE) +
  labs(
    title = "Minimum detectable effect size by atlas",
    subtitle = str_wrap(sprintf("Dataset-specific cells per embryo; points are mean interpolated MDEs for %.0f%% power; error bars are bootstrap SE", target_power * 100), width = 80),
    x = "Embryos per arm",
    y = "Minimum detectable\nfractional reduction",
    color = "Atlas"
  ) +
  theme_power(base_size = 18)

save_power_plot(
  p_mde,
  sprintf("minimum_detectable_effect_full_composition_%s_%shpf_%s_by_atlas", target_slug, analysis_timepoint, power_run_slug),
  width = 11,
  height = 7
)

p_mde